In [0]:
# Cargar datos limpios
from pyspark.sql.functions import col, avg, stddev, corr, count

df = spark.table("wine_quality_clean")

In [0]:
# Distribución de calidad por tipo de vino
quality_dist = df.groupBy("wine_type", "quality").count().orderBy("wine_type", "quality")
display(quality_dist)
# Después del display: clic en ícono de gráfico → Bar Chart
# X: quality, Y: count, Group: wine_type

wine_type,quality,count
red,3,10
red,4,53
red,5,577
red,6,535
red,7,167
red,8,17
white,3,20
white,4,153
white,5,1175
white,6,1788


In [0]:
# Promedios de variables químicas por calidad
from pyspark.sql.functions import round as spark_round

chemical_cols = ["fixed_acidity", "volatile_acidity", "citric_acid",
                 "residual_sugar", "chlorides", "alcohol", "sulphates", "pH"]

agg_exprs = [spark_round(avg(c), 3).alias(f"avg_{c}") for c in chemical_cols]
chem_by_quality = df.groupBy("quality").agg(*agg_exprs).orderBy("quality")
display(chem_by_quality)
# Visualizar como Line Chart: X = quality, Y = avg_alcohol (o cualquier variable)


quality,avg_fixed_acidity,avg_volatile_acidity,avg_citric_acid,avg_residual_sugar,avg_chlorides,avg_alcohol,avg_sulphates,avg_pH
3,7.853,0.517,0.281,5.14,0.077,10.215,0.506,3.258
4,7.304,0.462,0.272,4.035,0.061,10.215,0.507,3.236
5,7.333,0.394,0.306,5.482,0.066,9.872,0.529,3.214
6,7.169,0.316,0.325,5.153,0.054,10.649,0.534,3.224
7,7.122,0.292,0.336,4.171,0.045,11.511,0.55,3.24
8,6.82,0.303,0.341,4.772,0.04,11.912,0.519,3.24
9,7.42,0.298,0.386,4.12,0.027,12.18,0.466,3.308


In [0]:
# Correlación de variables con la calidad (pandas)
import pandas as pd

df_pd = df.toPandas()
correlaciones = df_pd.select_dtypes(include="number").corr()["quality"].sort_values(ascending=False)
print("Correlación de cada variable con 'quality':")
print(correlaciones)


Correlación de cada variable con 'quality':
quality                 1.000000
alcohol                 0.469422
citric_acid             0.097954
free_sulfur_dioxide     0.054002
sulphates               0.041884
pH                      0.039733
total_sulfur_dioxide   -0.050296
residual_sugar         -0.056830
fixed_acidity          -0.080092
chlorides              -0.202137
volatile_acidity       -0.265205
density                -0.326434
Name: quality, dtype: float64


In [0]:
# Comparativa de alcohol por tipo de vino y categoría
alcohol_stats = df.groupBy("wine_type", "quality_category").agg(
    spark_round(avg("alcohol"), 2).alias("avg_alcohol"),
    spark_round(stddev("alcohol"), 2).alias("std_alcohol"),
    count("*").alias("n_wines")
).orderBy("wine_type", "quality_category")
display(alcohol_stats)

wine_type,quality_category,avg_alcohol,std_alcohol,n_wines
red,Alta,11.55,1.01,184
red,Baja,10.22,0.92,63
red,Media,10.26,0.99,1112
white,Alta,11.58,1.14,825
white,Baja,10.21,1.04,173
white,Media,10.34,1.1,2963


In [0]:
# Proporción de vinos por categoría
cat_counts = df.groupBy("quality_category").count()
display(cat_counts)
# Visualizar como Pie Chart: Values = count, Group = quality_category

quality_category,count
Media,4075
Alta,1009
Baja,236
